In [18]:
import jax
import jax.numpy as jnp
from jax import jit, grad, vmap, jacobian, hessian

from functools import partial
print(jax.devices())

[CpuDevice(id=0)]


# Some Basic Modifications in JAX compared with numpy

In [ ]:
key = jax.random.PRNGKey(0)
key1, key2 = jax.random.split(key)

In [5]:
print(key1, key2, key)

[1797259609 2579123966] [ 928981903 3453687069] [0 0]


In [8]:
A = jnp.array([[1.0, 2.0], [3.0, 4.0]])
# A[0, 0] = 5.0  # Outputs ERROR: JAX arrays are immutable
A_new = A.at[0, 0].set(5.0)  # returns a new array

In [ ]:
# === JIT (Just In Time) compilation ===
def slow_fn(x):
    for _ in range(10):
        x = x @ x  # matrix power
    return x

fast_fn = jit(slow_fn)

_ = fast_fn(jnp.eye(2)) # Triggers traces + compilation process, subsequent calls will be faster

# Benchmark
%timeit slow_fn(jnp.eye(100)).block_until_ready()
%timeit fast_fn(jnp.eye(100)).block_until_ready()

595 μs ± 14.3 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
416 μs ± 15.1 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [ ]:
# === Automatic differentiation ===
# Scalar function: gradient is straightforward
def f(x):
    return jnp.sum(x ** 2)

grad_f = grad(f)
print(grad_f(jnp.array([3.0, 4.0])))

[6. 8.]


In [12]:
# Matrix-valued: need to differentiate scalar-valued objectives, and gradient of log-determinant (important for Gaussian likelihoods)
def log_det(L):
    """L is a lower triangular Cholesky factor."""
    return 2.0 * jnp.sum(jnp.log(jnp.diag(L)))

L = jnp.array([[2.0, 0.0], [1.0, 3.0]])
print(grad(log_det)(L)) # d/dL log|LL^T| = 2 * L^{-T} for diagonal, off-diag entries are 0

[[1.        0.       ]
 [0.        0.6666667]]


In [16]:
# === VMAP: Automatic vectorization ===
def matvec(A, x):
    return A @ x

batch_matvec = vmap(matvec, in_axes=(None, 0))  # Apply to a batch of vectors simultaneously. Where, A fixed, x batched along axis 0

A = jnp.eye(3)
X = jax.random.normal(jax.random.PRNGKey(0), (100, 3))
result = batch_matvec(A, X)  # shape: (100, 3)
result.shape

(100, 3)

In [17]:
# JAX PRNG 
key = jax.random.PRNGKey(42)

# WRONG: reusing the same key gives identical samples
x1 = jax.random.normal(key, (3,))
x2 = jax.random.normal(key, (3,))
print(jnp.allclose(x1, x2))  # True — same key, same output!

# RIGHT: split the key
key, subkey1, subkey2 = jax.random.split(key, 3)
x1 = jax.random.normal(subkey1, (3,))
x2 = jax.random.normal(subkey2, (3,))
print(jnp.allclose(x1, x2))  # False — different subkeys

# Pattern for loops:
key = jax.random.PRNGKey(0)
samples = []
for i in range(5):
    key, subkey = jax.random.split(key)
    samples.append(jax.random.normal(subkey, (2,)))

# Better: vectorized with vmap
keys = jax.random.split(jax.random.PRNGKey(0), 1000)
samples = vmap(lambda k: jax.random.normal(k, (2,)))(keys)
# shape: (1000, 2)

True
False


In [20]:
# Jacobian of a vector-valued function
def g(x):
    return jnp.array([x[0]**2 + x[1], jnp.sin(x[0]) * x[1]])

x0 = jnp.array([1.0, 2.0])
J = jacobian(g)(x0) # Row i, col j = dg_i / dx_j; [[2*x0, 1], [cos(x0)*x1, sin(x0)]]
print("Jacobian shape:", J.shape)
print(J)

# Hessian of a scalar function
def h(x):
    return x[0]**3 + x[0]*x[1]**2

H = hessian(h)(x0) # [[6*x0, 2*x1], [2*x1, 2*x0]]
print("Hessian shape:", H.shape)
print(H)

Jacobian shape: (2, 2)
[[2.         1.        ]
 [1.0806046  0.84147096]]
Hessian shape: (2, 2)
[[6. 4.]
 [4. 2.]]
